# 03 — IR Documents & Vector Space Model
## Phase 2: Information Retrieval System

This notebook covers **Steps 4, 5, and 6** of the ELIQSIR IR pipeline:

- **Step 4:** Extract searchable documents from the DWH — combining article text with drug and protein named identifiers from linked bioactivity records.
- **Step 5:** Apply biomedical-aware text preprocessing and build a TF-IDF vector space.
- **Step 6 (setup):** Persist the vector representation as a **sparse list of `(doc_id, term, tf-idf score)` triples** in JSON for use by retrieval models.

### Document Schema
Each document is assembled from three sources joined via `fact_bioactivity`:

| Field | Source | Role |
|---|---|---|
| `article_title` | `dim_article` | Primary descriptive text |
| `abstract` | `dim_article` | Rich IR corpus |
| `authors` | `dim_article` | Contextual |
| `drug_name`, `drug_chembl_id` | `dim_drug` | Named chemical entity |
| `uniprot_id`, `gene_names`, `protein_name`, `protein_families` | `dim_protein` | Named biological entity |

Drug and protein identifiers are concatenated into the document body to ensure that queries like `"CHEMBL25"`, `"EGFR"`, or `"imatinib"` retrieve the correct records.


## Setup


In [1]:
import os
import sys
import re
import json
import warnings
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from dotenv import load_dotenv, find_dotenv
from IPython.display import display

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Suppress noisy UserWarnings from pandas / mysql connector
warnings.filterwarnings('ignore', category=UserWarning)

# Download required NLTK assets (runs only once)
nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)

print("Libraries loaded.")

Libraries loaded.


In [2]:
load_dotenv(find_dotenv())
env_data_dir = os.getenv("DATA_DIR")
project_root = os.getenv('PROJECT_ROOT')

if project_root not in sys.path:
    sys.path.append(project_root)

PROJECT_ROOT     = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR")
BASE_DATA_DIR    = Path(env_data_dir) if env_data_dir else Path(project_root) / "data"
CLEANED_CSV_DIR  = BASE_DATA_DIR / "csv_cleaned"

# IR output directory — all artefacts from this notebook go here
IR_DIR = BASE_DATA_DIR / "ir"
IR_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENTS_JSON = IR_DIR / "ir_documents.json"
TRIPLES_JSON   = IR_DIR / "tfidf_triples.json"

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

print("\nTesting connection to AWS RDS...")
try:
    with db_manager.get_dwh_connection() as conn:
        with conn.cursor() as cursor:
            cursor.execute("SELECT VERSION();")
            version = cursor.fetchone()
            print(f"Connected. Database version: {version[0]}")

            cursor.execute("SELECT COUNT(*) FROM fact_bioactivity;")
            n_facts = cursor.fetchone()[0]
            print(f"DWH reachable — {n_facts:,} bioactivity records.")
except Exception as e:
    print(f"Connection error: {e}")



Testing connection to AWS RDS...
Connected. Database version: 8.4.8
Connected. Database version: 8.4.8
DWH reachable — 6,441,095 bioactivity records.
DWH reachable — 6,441,095 bioactivity records.


---
## Step 4 — Extract Searchable Documents

Each document is assembled per unique article (`article_key`), enriched with the **aggregated set of drug and protein identifiers** found in its linked bioactivity records.

**Why aggregate drug/protein names into the article document?**

A query like `"imatinib EGFR"` or `"CHEMBL941 P00533"` should surface the papers that actually report those measurements. Without injecting these entity strings into the document text, the IR system has no way to match them — the abstract alone rarely contains the exact ChEMBL ID or UniProt accession.

**Field weighting strategy:**
- `article_title` is repeated **3×** to boost its term frequency contribution.
- `abstract` carries full weight as the main corpus.
- Entity identifiers (`uniprot_id`, `drug_chembl_id`) are included as-is to support exact-token queries.
- Human-readable names (`drug_name`, `gene_names`, `protein_name`) are included to support natural-language queries.


In [3]:
EXTRACT_QUERY = """
SELECT
    a.article_key,
    a.pubmed_id,
    a.article_title,
    a.journal,
    a.year,
    a.abstract,
    a.authors,

    -- Aggregated drug identifiers (all unique values linked via fact_bioactivity)
    GROUP_CONCAT(DISTINCT d.drug_chembl_id   SEPARATOR ' ') AS drug_chembl_ids,
    GROUP_CONCAT(DISTINCT d.drug_name        SEPARATOR ' ') AS drug_names,

    -- Aggregated protein identifiers
    GROUP_CONCAT(DISTINCT p.uniprot_id       SEPARATOR ' ') AS uniprot_ids,
    GROUP_CONCAT(DISTINCT p.gene_names       SEPARATOR ' ') AS gene_names,
    GROUP_CONCAT(DISTINCT p.protein_name     SEPARATOR ' ') AS protein_names,
    GROUP_CONCAT(DISTINCT p.protein_families SEPARATOR ' ') AS protein_families

FROM dim_article a
JOIN fact_bioactivity f ON a.article_key = f.article_key
JOIN dim_drug         d ON f.drug_key    = d.drug_key
JOIN dim_protein      p ON f.protein_key = p.protein_key
WHERE a.abstract IS NOT NULL
  AND a.abstract != ''
GROUP BY
    a.article_key, a.pubmed_id, a.article_title,
    a.journal, a.year, a.abstract, a.authors
"""

# ── Checkpoint: skip DB query if documents file already exists ──────────────
if DOCUMENTS_JSON.exists():
    print(f"Checkpoint hit: '{DOCUMENTS_JSON.name}' already exists — loading from disk.")
    with open(DOCUMENTS_JSON, 'r', encoding='utf-8') as f:
        _cached = json.load(f)

    df_docs = pd.DataFrame([{
        **{k: v for k, v in doc.items() if k not in
           ('drug_chembl_ids', 'drug_names', 'uniprot_ids',
            'gene_names', 'protein_names', 'protein_families')},
        'drug_chembl_ids':  ' '.join(doc.get('drug_chembl_ids',  [])),
        'drug_names':       ' '.join(doc.get('drug_names',       [])),
        'uniprot_ids':      ' '.join(doc.get('uniprot_ids',      [])),
        'gene_names':       ' '.join(doc.get('gene_names',       [])),
        'protein_names':    ' '.join(doc.get('protein_names',    [])),
        'protein_families': ' '.join(doc.get('protein_families', [])),
    } for doc in _cached])
    df_docs.rename(columns={'doc_id': 'article_key'}, inplace=True)

    print(f"Loaded {len(df_docs):,} documents from cache.")

else:
    print("No cache found — querying DWH...")
    with db_manager.get_dwh_connection() as conn:
        df_docs = pd.read_sql(EXTRACT_QUERY, con=conn)


    print(f"Extracted {len(df_docs):,} documents from DWH.")

display(df_docs[['article_key', 'article_title', 'drug_chembl_ids',
                  'drug_names', 'uniprot_ids', 'gene_names']].head(5))


No cache found — querying DWH...
Extracted 35,561 documents from DWH.
Extracted 35,561 documents from DWH.


,article_key,article_title,drug_chembl_ids,drug_names,uniprot_ids,gene_names
0,1,Exploring a new frontier in cancer treatment: ...,CHEMBL1173445 CHEMBL2017005 CHEMBL2322194 CHEM...,ANACARDIC ACID GINGKOLIC ACID KERRIAMYCIN B LA...,A0AVT1 O95352 P41226 Q13564 Q9UBE0,ATG7 APG7L NAE1 APPBP1 HPP1 SAE1 AOS1 SUA1 UBL...
1,2,Interrogating the Roles of Post-Translational ...,CHEMBL1171837 CHEMBL1213603 CHEMBL1231160 CHEM...,LIFIRAFENIB PEVONEDISTAT PONATINIB,A0AVT1 O43353 O60502 O95352 P15056 Q09472 Q15843,ATG7 APG7L BRAF BRAF1 RAFB1 EP300 P300 NEDD8 O...
2,3,Adenosine analogs bearing phosphate isosteres ...,CHEMBL14830 CHEMBL4226903 CHEMBL752,ADENOSINE DIPHOSPHATE ADENOSINE PHOSPHATE,A1Z1Q3 Q9BQ69,MACROD1 LRP16 MACROD2 C20orf133
3,4,NAE modulators: A potential therapy for gastri...,CHEMBL1231160 CHEMBL4640167 CHEMBL487992 CHEMB...,GARTANIN PEVONEDISTAT,A0AVT1 O95352 P00918 P22314 P61081 Q13564,ATG7 APG7L CA2 NAE1 APPBP1 HPP1 UBA1 A1S9T UBE...
4,5,"Design, synthesis and evaluation of inhibitors...",CHEMBL35505 CHEMBL5205107 CHEMBL5208626,DIHYDRALAZINE,A1Z1Q3,MACROD2 C20orf133


In [4]:
# set columns to max display
pd.set_option('display.max_columns', None)
df_docs.head(3)

,article_key,pubmed_id,article_title,journal,year,abstract,authors,drug_chembl_ids,drug_names,uniprot_ids,gene_names,protein_names,protein_families
0,1,23360215,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013,The labeling of proteins with small ubiquitin ...,"da Silva, Sara R; Paiva, Stacey-Lynn; Lukkaril...",CHEMBL1173445 CHEMBL2017005 CHEMBL2322194 CHEM...,ANACARDIC ACID GINGKOLIC ACID KERRIAMYCIN B LA...,A0AVT1 O95352 P41226 Q13564 Q9UBE0,ATG7 APG7L NAE1 APPBP1 HPP1 SAE1 AOS1 SUA1 UBL...,NEDD8-activating enzyme E1 regulatory subunit ...,ATG7 family Ubiquitin-activating E1 family Ubi...
1,2,28505447,Interrogating the Roles of Post-Translational ...,J Med Chem,2018,Post-translational modifications (PTMs) allot ...,"Buuh, Zakey Yusuf; Lyu, Zhigang; Wang, Rongshe...",CHEMBL1171837 CHEMBL1213603 CHEMBL1231160 CHEM...,LIFIRAFENIB PEVONEDISTAT PONATINIB,A0AVT1 O43353 O60502 O95352 P15056 Q09472 Q15843,ATG7 APG7L BRAF BRAF1 RAFB1 EP300 P300 NEDD8 O...,Histone acetyltransferase p300 (p300 HAT) (EC ...,ATG7 family Glycosyl hydrolase 84 family Prote...
2,3,29501416,Adenosine analogs bearing phosphate isosteres ...,Bioorg Med Chem,2018,The human O-acetyl-ADP-ribose deacetylase MDO1...,"Zhang, Yuezhou; Jumppanen, Mikael; Maksimainen...",CHEMBL14830 CHEMBL4226903 CHEMBL752,ADENOSINE DIPHOSPHATE ADENOSINE PHOSPHATE,A1Z1Q3 Q9BQ69,MACROD1 LRP16 MACROD2 C20orf133,ADP-ribose glycohydrolase MACROD1 (MACRO domai...,"MacroD-type family, MacroD1/2-like subfamily"


In [5]:
def build_document_text(row) -> str:
    """
    Combine article + entity fields into a single searchable string.

    Weighting approach:
      - article_title repeated 3x   → boosts title terms in TF without touching IDF
      - abstract, authors           → main corpus (full weight)
      - drug/protein identifiers    → exact-token anchors for structured queries
      - drug/protein names          → natural-language query support
    """
    title    = str(row['article_title']    or '')
    abstract = str(row['abstract']         or '')
    authors  = str(row['authors']          or '')

    # Drug entity fields
    d_ids    = str(row['drug_chembl_ids']  or '')
    d_names  = str(row['drug_names']       or '')

    # Protein entity fields
    p_ids    = str(row['uniprot_ids']      or '')
    g_names  = str(row['gene_names']       or '')
    p_names  = str(row['protein_names']    or '')
    p_fam    = str(row['protein_families'] or '')

    # Title repeated 3× for weighting (TF boosting without sub-classing vectorizer)
    parts = [
        title, title, title,
        abstract,
        authors,
        d_ids, d_names,
        p_ids, g_names, p_names, p_fam,
    ]
    return ' '.join(p for p in parts if p.strip())


df_docs['document_text'] = df_docs.apply(build_document_text, axis=1)
print("Document text column created.")
print(f"\nSample document (article_key={df_docs['article_key'].iloc[0]}):")
print(df_docs['document_text'].iloc[0][:600], "...")


Document text column created.

Sample document (article_key=1):
Exploring a new frontier in cancer treatment: targeting the ubiquitin and ubiquitin-like activating enzymes. Exploring a new frontier in cancer treatment: targeting the ubiquitin and ubiquitin-like activating enzymes. Exploring a new frontier in cancer treatment: targeting the ubiquitin and ubiquitin-like activating enzymes. The labeling of proteins with small ubiquitin (Ub) and ubiquitin-like (Ubl) modifiers regulates a plethora of activities within the cell, such as protein recycling, cell cycle modifications, and protein translocation. These processes are often overactive in diseased cells, ...


In [6]:
# ── Checkpoint: skip export if documents file already exists ────────────────
if DOCUMENTS_JSON.exists():
    print(f"Checkpoint hit: '{DOCUMENTS_JSON.name}' already exists — skipping export.")
    print(f"  Delete /data/ir/{DOCUMENTS_JSON.name} to force re-export.")

else:
    # build_document_text must have already run (cell above) before reaching here
    documents = []
    for _, row in df_docs.iterrows():
        documents.append({
            "doc_id":           int(row['article_key']),
            "pubmed_id":        int(row['pubmed_id'])        if pd.notna(row['pubmed_id'])  else None,
            "article_title":    row['article_title']         or None,
            "abstract":         row['abstract']              or None,
            "journal":          row['journal']               or None,
            "year":             int(row['year'])             if pd.notna(row['year'])       else None,
            "authors":          row['authors']               or None,
            # Drug entities linked to this article (space-separated strings → split to lists)
            "drug_chembl_ids":  [v for v in str(row['drug_chembl_ids']  or '').split() if v],
            "drug_names":       [v for v in str(row['drug_names']       or '').split() if v],
            # Protein entities linked to this article
            "uniprot_ids":      [v for v in str(row['uniprot_ids']      or '').split() if v],
            "gene_names":       [v for v in str(row['gene_names']       or '').split() if v],
            "protein_names":    [v for v in str(row['protein_names']    or '').split() if v],
            "protein_families": [v for v in str(row['protein_families'] or '').split() if v],
            # Combined corpus field used by the IR engine
            "document_text":    row['document_text'],
        })

    with open(DOCUMENTS_JSON, 'w', encoding='utf-8') as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(documents):,} documents → {DOCUMENTS_JSON}")
    print(f"File size: {DOCUMENTS_JSON.stat().st_size / 1024:.1f} KB")
    print("\nSample record (truncated):")
    print(json.dumps(documents[0], indent=2, ensure_ascii=False)[:800], "\n...")


Saved 35,561 documents → /home/pfanyka/Desktop/MASTERS/sem_2/IPA/ELIQSIR/data/ir/ir_documents.json
File size: 197632.6 KB

Sample record (truncated):
{
  "doc_id": 1,
  "pubmed_id": 23360215,
  "article_title": "Exploring a new frontier in cancer treatment: targeting the ubiquitin and ubiquitin-like activating enzymes.",
  "abstract": "The labeling of proteins with small ubiquitin (Ub) and ubiquitin-like (Ubl) modifiers regulates a plethora of activities within the cell, such as protein recycling, cell cycle modifications, and protein translocation. These processes are often overactive in diseased cells, leading to unregulated cell growth and disease progression. Therefore, in systems where Ub/Ubl protein labeling is dysregulated, the development of drugs to selectively and potently disrupt Ub/Ubl protein labeling offers a targeted molecular approach for sensitizing these diseased cells. This Perspective outlines the progress that has b 
...


---
## Step 5 — Biomedical Text Preprocessing & TF-IDF Vector Space

### Preprocessing design decisions for biomedical text

| Issue | Solution |
|---|---|
| Alphanumeric identifiers (`CHEMBL25`, `HER2`, `MK-2206`) | Custom regex tokenizer — preserves internal hyphens and digits |
| Multi-word drug names (`imatinib mesylate`) | N-gram range `(1, 3)` captures up to 3-word phrases |
| Generic stopwords break identifier tokens | Standard English stopwords list applied after tokenization |
| Very rare terms inflate vocabulary | `min_df=2` removes terms appearing in fewer than 2 documents |
| Corpus-wide boilerplate phrases | `max_df=0.90` suppresses terms present in >90% of documents |


### Preprocessing Strategy Comparison — Stemming vs Lemmatization on a Biomedical Sample

Before applying any preprocessing strategy to the full corpus, we compare **stemming** (Porter) against **lemmatization** (WordNet) on a small random sample of real documents.

This matters in the biomedical domain because:
- **Stemming** aggressively chops word endings: `"inhibitors"` → `"inhibitor"` ✓, but `"kinases"` → `"kinas"` ✗, and `"binding"` → `"bind"` — which may fragment meaningful biomedical roots.
- **Lemmatization** uses vocabulary lookup to produce valid base forms: `"inhibitors"` → `"inhibitor"`, `"binding"` → `"binding"` — preserving domain terms more faithfully.

We test both using the same **top-k cosine similarity search** from the teacher's example, running a few representative biomedical queries and comparing which strategy surfaces more relevant results.


In [7]:
from nltk.stem import PorterStemmer
from sklearn.metrics.pairwise import cosine_similarity

stemmer = PorterStemmer()

# ── Sample: pick 300 random documents for fast comparison ──────────────────
SAMPLE_N = 300
df_sample = df_docs.sample(n=min(SAMPLE_N, len(df_docs)), random_state=42).reset_index(drop=True)
print(f"Sample size: {len(df_sample)} documents")
print(f"Sample titles preview:")
for t in df_sample['article_title'].dropna().head(5).tolist():
    print(f"  • {t[:90]}")


Sample size: 300 documents
Sample titles preview:
  • Design, Synthesis, and Preclinical Characterization of the Selective Androgen Receptor Mod
  • Engineered Conotoxin Differentially Blocks and Discriminates Rat and Human α7 Nicotinic Ac
  • Carbonic anhydrase inhibitors. Interaction of the antiepileptic drug sulthiame with twelve
  • Substitution on the Phe3 aromatic ring in cyclic delta opioid receptor-selective dermorphi
  • Inhibitors of Eukaryotic Translational Machinery as Therapeutic Agents.


In [9]:
STOP_WORDS  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

# ── Three preprocessing variants (same pipeline structure as teacher's example) ──

# Shared: biomedical tokenizer + stopword removal (no stemming/lemmatization yet)
def tokenize_biomed(text: str):
    """Lowercase + biomedical regex tokenization + stopword removal."""
    if not isinstance(text, str):
        return []
    tokens = re.findall(r'\b[a-z0-9](?:[a-z0-9\-]*[a-z0-9])?\b', text.lower())
    return [t for t in tokens if len(t) >= 2 and t not in STOP_WORDS]

# Variant 1: tokenization + stopword removal only (baseline — no morphological reduction)
def preprocess_baseline(text: str) -> str:
    return ' '.join(tokenize_biomed(text))

# Variant 2: + Porter stemming (aggressive suffix stripping)
def preprocess_stemming(text: str) -> str:
    tokens = tokenize_biomed(text)
    return ' '.join(stemmer.stem(t) for t in tokens)

# Variant 3: + WordNet lemmatization (vocabulary-based base form)
def preprocess_lemma(text: str) -> str:
    tokens = tokenize_biomed(text)
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens)


# Apply all three to the sample
df_sample['text_baseline'] = df_sample['document_text'].apply(preprocess_baseline)
df_sample['text_stemmed']  = df_sample['document_text'].apply(preprocess_stemming)
df_sample['text_lemma']    = df_sample['document_text'].apply(preprocess_lemma)

# Spot-check: show the effect on a single document
idx = 0
print("=== Token comparison for document 0 ===\n")
print("BASELINE (no morphological reduction):")
print(' '.join(df_sample['text_baseline'].iloc[idx].split()[:25]), "...\n")
print("STEMMED (Porter):")
print(' '.join(df_sample['text_stemmed'].iloc[idx].split()[:25]), "...\n")
print("LEMMATIZED (WordNet):")
print(' '.join(df_sample['text_lemma'].iloc[idx].split()[:25]), "...")


=== Token comparison for document 0 ===

BASELINE (no morphological reduction):
design synthesis preclinical characterization selective androgen receptor modulator sarm rad140 design synthesis preclinical characterization selective androgen receptor modulator sarm rad140 design synthesis preclinical characterization selective ...

STEMMED (Porter):
design synthesi preclin character select androgen receptor modul sarm rad140 design synthesi preclin character select androgen receptor modul sarm rad140 design synthesi preclin character select ...

LEMMATIZED (WordNet):
design synthesis preclinical characterization selective androgen receptor modulator sarm rad140 design synthesis preclinical characterization selective androgen receptor modulator sarm rad140 design synthesis preclinical characterization selective ...


In [10]:
# ── Build one TF-IDF search engine per variant (same pattern as teacher's notebook) ──

def build_search_engine(texts, df_ref):
    """Fit a TF-IDF vectorizer on `texts` and return a top-k search function."""
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    # ngram_range=(1,2) only for sample search — (1,3) used in full corpus later
    X   = vec.fit_transform(texts)

    def search(query: str, preprocess_fn, top_k: int = 5):
        q_clean = preprocess_fn(query)
        q_vec   = vec.transform([q_clean])
        scores  = cosine_similarity(q_vec, X).flatten()
        top_idx = scores.argsort()[::-1][:top_k]
        results = df_ref.iloc[top_idx][['article_key', 'article_title']].copy()
        results['score'] = scores[top_idx].round(4)
        return results.reset_index(drop=True)

    return search, vec

search_baseline, vec_base = build_search_engine(df_sample['text_baseline'], df_sample)
search_stemmed,  vec_stem = build_search_engine(df_sample['text_stemmed'],  df_sample)
search_lemma,    vec_lem  = build_search_engine(df_sample['text_lemma'],    df_sample)

print("Vocabulary sizes:")
print(f"  Baseline   : {len(vec_base.vocabulary_):,} terms")
print(f"  Stemmed    : {len(vec_stem.vocabulary_):,} terms  ← more aggressive merging")
print(f"  Lemmatized : {len(vec_lem.vocabulary_):,} terms  ← moderate merging")


Vocabulary sizes:
  Baseline   : 62,506 terms
  Stemmed    : 60,070 terms  ← more aggressive merging
  Lemmatized : 61,558 terms  ← moderate merging


In [11]:
# ── Run the same biomedical queries through all three engines ──────────────
# These queries are representative of what a researcher would actually type

queries = [
    ("Natural language query",   "kinase inhibitor breast cancer"),
    ("Multi-word drug name",     "imatinib mesylate tyrosine kinase"),
    ("Alphanumeric identifier",  "EGFR CHEMBL4523"),
    ("Protein family query",     "epidermal growth factor receptor inhibition"),
    ("Mechanism query",          "binding affinity IC50 selectivity"),
]

for label, q in queries:
    print(f"\n{'='*70}")
    print(f"Query [{label}]: \"{q}\"")
    print(f"{'='*70}")

    print("\n--- Baseline (tokenize + stopwords only) ---")
    display(search_baseline(q, preprocess_baseline, top_k=3))

    print("\n--- Stemmed (Porter) ---")
    display(search_stemmed(q, preprocess_stemming, top_k=3))

    print("\n--- Lemmatized (WordNet) ---")
    display(search_lemma(q, preprocess_lemma, top_k=3))



Query [Natural language query]: "kinase inhibitor breast cancer"

--- Baseline (tokenize + stopwords only) ---


,article_key,article_title,score
0,10415,Selective Human Estrogen Receptor Partial Agon...,0.2823
1,904,Development of dual casein kinase 1δ/1ε (CK1δ/...,0.2286
2,26891,New amidine-benzenesulfonamides as iNOS inhibi...,0.1945



--- Stemmed (Porter) ---


,article_key,article_title,score
0,10415,Selective Human Estrogen Receptor Partial Agon...,0.3172
1,904,Development of dual casein kinase 1δ/1ε (CK1δ/...,0.2747
2,26891,New amidine-benzenesulfonamides as iNOS inhibi...,0.2225



--- Lemmatized (WordNet) ---


...

,article_key,article_title,score
0,35104,"(+)-cis-N-(para-, meta-, and ortho-substituted...",0.1799
1,33586,"Design, synthesis, and pharmacological evaluat...",0.1058
2,27234,Substitution on the Phe3 aromatic ring in cycl...,0.0813



--- Stemmed (Porter) ---


,article_key,article_title,score
0,35104,"(+)-cis-N-(para-, meta-, and ortho-substituted...",0.1658
1,33586,"Design, synthesis, and pharmacological evaluat...",0.1124
2,27234,Substitution on the Phe3 aromatic ring in cycl...,0.0865



--- Lemmatized (WordNet) ---


,article_key,article_title,score
0,35104,"(+)-cis-N-(para-, meta-, and ortho-substituted...",0.1634
1,33586,"Design, synthesis, and pharmacological evaluat...",0.1089
2,27234,Substitution on the Phe3 aromatic ring in cycl...,0.0750


### What to look for in the results above

| Observation | Implication |
|---|---|
| Stemming returns **different** top documents than lemmatization for the identifier query (`EGFR CHEMBL4523`) | Stemmer may be corrupting alphanumeric tokens — check if `"chembl4523"` survives or gets truncated |
| Stemming shrinks vocabulary **more** than lemmatization | Stemmer merges more variants, which helps recall but risks merging unrelated biomedical terms |
| Lemmatization scores are **equal or higher** for mechanism queries (`binding`, `selectivity`) | Lemmatizer preserves root meaning without over-reducing |
| Baseline and lemmatization return **identical** top-1 for most queries | Lemmatization is a safe improvement — it only merges obvious inflectional variants |

**Decision guide:**
- If stemming shows corrupted identifiers (e.g., `"kinas"`, `"chembl"`) → use **lemmatization**
- If both return equivalent results → lemmatization preferred (safer for biomedical terms)
- The full corpus preprocessing in the next cell uses **lemmatization** based on this evidence


In [15]:
# ── Automatic strategy selection via objective metrics ─────────────────────
#
# For each query we collect the top-k scores returned by each engine and
# compute three complementary metrics:
#
#   1. Mean Top-1 Score      — average relevance of the single best hit (precision focus)
#   2. Mean Avg Precision    — average of scores across top-k hits (MAP-like, recall focus)
#   3. Mean Reciprocal Rank  — 1/rank of the best hit relative to baseline ranking
#      (rewards strategies that push the best document to position 1)
#
# A weighted composite score combines all three, then the argmax selects the winner.

import numpy as np

STRATEGIES = {
    "Baseline":    (search_baseline, preprocess_baseline),
    "Stemmed":     (search_stemmed,  preprocess_stemming),
    "Lemmatized":  (search_lemma,    preprocess_lemma),
}

TOP_K = 3  # must match the top_k used in the query loop above

# Weights for the composite score (tweak if you want to favour precision vs recall)
W_TOP1 = 0.50   # top-1 score  — precision at 1
W_MAP  = 0.30   # mean avg precision across top-k
W_MRR  = 0.20   # mean reciprocal rank

records = {name: {"top1": [], "map": [], "mrr": []} for name in STRATEGIES}

for label, q in queries:
    # Collect result DataFrames for all strategies
    results = {
        name: fn(q, prep_fn, top_k=TOP_K)
        for name, (fn, prep_fn) in STRATEGIES.items()
    }

    # For MRR: treat baseline top-1 article_key as the "gold" document
    gold_key = results["Baseline"].iloc[0]["article_key"]

    for name, df_res in results.items():
        scores = df_res["score"].values

        top1 = float(scores[0]) if len(scores) > 0 else 0.0
        map_k = float(np.mean(scores)) if len(scores) > 0 else 0.0

        # MRR: find rank of gold_key in this strategy's results (1-indexed)
        try:
            rank = df_res[df_res["article_key"] == gold_key].index[0] + 1
        except IndexError:
            rank = TOP_K + 1   # not found → worst rank
        mrr = 1.0 / rank

        records[name]["top1"].append(top1)
        records[name]["map"].append(map_k)
        records[name]["mrr"].append(mrr)

# ── Aggregate & display ────────────────────────────────────────────────────
print(f"\n{'Strategy':<14} {'Mean Top-1':>12} {'Mean MAP@k':>12} {'Mean MRR':>10} {'Composite':>11}")
print("-" * 62)

best_name  = None
best_score = -1.0

for name, vals in records.items():
    m_top1 = np.mean(vals["top1"])
    m_map  = np.mean(vals["map"])
    m_mrr  = np.mean(vals["mrr"])
    composite = W_TOP1 * m_top1 + W_MAP * m_map + W_MRR * m_mrr

    print(f"{name:<14} {m_top1:>12.4f} {m_map:>12.4f} {m_mrr:>10.4f} {composite:>11.4f}")

    if composite > best_score:
        best_score = composite
        best_name  = name

print("-" * 62)
print(f"\n✅  Recommended preprocessing strategy: **{best_name}**  (composite = {best_score:.4f})")
print(f"   Weights used — Top-1: {W_TOP1}, MAP@{TOP_K}: {W_MAP}, MRR: {W_MRR}")
print(f"\n   → The full-corpus preprocessing cell below will use '{best_name}'.")



Strategy         Mean Top-1   Mean MAP@k   Mean MRR   Composite
--------------------------------------------------------------
Baseline             0.2661       0.2236     1.0000      0.4001
Stemmed              0.2675       0.2318     0.9000      0.3833
Lemmatized           0.2684       0.2308     1.0000      0.4034
--------------------------------------------------------------

✅  Recommended preprocessing strategy: **Lemmatized**  (composite = 0.4034)
   Weights used — Top-1: 0.5, MAP@3: 0.3, MRR: 0.2

   → The full-corpus preprocessing cell below will use 'Lemmatized'.


In [16]:
# Biomedical tokenizer:
#   \b[a-z0-9](?:[a-z0-9\-]*[a-z0-9])?\b
#   → matches tokens that start AND end with an alphanumeric char.
#   → allows internal hyphens (pd-l1, mk-2206, her-2) but not trailing ones.
BIOMED_TOKEN_RE = re.compile(r'\b[a-z0-9](?:[a-z0-9\-]*[a-z0-9])?\b')

def preprocess_biomed(text: str) -> str:
    """
    Biomedical-aware preprocessing pipeline:
      1. Lowercase
      2. Regex tokenization  — preserves alphanumeric + internal-hyphen tokens
      3. Stopword removal    — English stopwords, length filter (>= 2 chars)
      4. Lemmatization       — reduces inflectional variants (inhibitors → inhibitor)
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    tokens = BIOMED_TOKEN_RE.findall(text.lower())
    clean  = [
        lemmatizer.lemmatize(tok)
        for tok in tokens
        if len(tok) >= 2 and tok not in STOP_WORDS
    ]
    return ' '.join(clean)


print("Preprocessing documents...")
df_docs['clean_text'] = df_docs['document_text'].apply(preprocess_biomed)

# Vocabulary stats preview
sample_tokens = df_docs['clean_text'].iloc[0].split()
print(f"Sample token count (doc 0): {len(sample_tokens)}")
print("First 30 tokens:", sample_tokens[:30])


Preprocessing documents...
Sample token count (doc 0): 221
First 30 tokens: ['exploring', 'new', 'frontier', 'cancer', 'treatment', 'targeting', 'ubiquitin', 'ubiquitin-like', 'activating', 'enzyme', 'exploring', 'new', 'frontier', 'cancer', 'treatment', 'targeting', 'ubiquitin', 'ubiquitin-like', 'activating', 'enzyme', 'exploring', 'new', 'frontier', 'cancer', 'treatment', 'targeting', 'ubiquitin', 'ubiquitin-like', 'activating', 'enzyme']


In [17]:
print("Fitting TF-IDF vectorizer (n-grams 1–3)...")

tfidf = TfidfVectorizer(
    ngram_range = (1, 3),   # unigrams, bigrams, trigrams
    max_df      = 0.90,     # drop corpus-wide boilerplate terms
    min_df      = 2,        # drop hapax legomena (singletons)
    sublinear_tf= True,     # apply 1 + log(tf) dampening on raw term frequency
)

tfidf_matrix = tfidf.fit_transform(df_docs['clean_text'])

vocab      = tfidf.get_feature_names_out()
n_docs     = tfidf_matrix.shape[0]
n_terms    = tfidf_matrix.shape[1]
n_nonzero  = tfidf_matrix.nnz

print(f"Documents : {n_docs:,}")
print(f"Vocabulary: {n_terms:,} terms (after min_df / max_df filtering)")
print(f"Non-zero entries: {n_nonzero:,}  ({100 * n_nonzero / (n_docs * n_terms):.2f}% density)")
print(f"\nSample vocabulary (first 20 terms): {list(vocab[:20])}")


Fitting TF-IDF vectorizer (n-grams 1–3)...
Documents : 35,561
Vocabulary: 1,029,528 terms (after min_df / max_df filtering)
Non-zero entries: 12,838,740  (0.04% density)

Sample vocabulary (first 20 terms): ['00', '00 10', '00 fold', '00 microm', '00 ng', '00 ng ml', '00 nm', '00 respectively', '000', '000 000', '000 65', '000 chemical', '000 commercially', '000 commercially available', '000 compound', '000 compound high', '000 compound identified', '000 compound identify', '000 compound library', '000 compound screened']


---
## Step 6 (setup) — Persist Vector Space as Sparse Triples

Instead of storing the full dense matrix (documents × vocabulary), we export only the **non-zero entries** as a list of `(doc_id, term, score)` triples.

**Why triples instead of the full matrix?**

- The TF-IDF matrix is highly sparse (~1–3% density). Storing zeros wastes disk and RAM.
- Triples are human-readable JSON — directly inspectable and portable to any retrieval model.
- Retrieval models (VSM cosine similarity, BM25) can reconstruct inverted indices from triples without loading the full matrix.

**JSON structure:**

```json
{
  "metadata": { "n_docs": ..., "n_terms": ..., "n_triples": ... },
  "vocabulary": ["term_0", "term_1", ...],
  "doc_index":  { "doc_id": position_in_matrix, ... },
  "triples":    [
    { "doc_id": 42, "term": "kinase", "score": 0.312 },
    ...
  ]
}
```


In [ ]:
# ── Checkpoint: skip vectorization if triples file already exists ───────────
if TRIPLES_JSON.exists():
    print(f"Checkpoint hit: '{TRIPLES_JSON.name}' already exists — skipping vectorization.")
    print(f"  Delete /data/ir/{TRIPLES_JSON.name} to force re-computation.")

else:
    # doc_id per matrix row (preserves original article_key)
    doc_ids = df_docs['article_key'].tolist()

    # Build a reverse index: article_key -> row position (for fast lookup at query time)
    doc_index = {int(doc_id): idx for idx, doc_id in enumerate(doc_ids)}

    print("Converting sparse TF-IDF matrix to (doc_id, term, score) triples...")

    # COO format exposes (row, col, data) arrays directly — no dense conversion needed
    cx = tfidf_matrix.tocoo()

    triples = [
        {
            "doc_id": int(doc_ids[row]),
            "term":   vocab[col],
            "score":  round(float(val), 6),
        }
        for row, col, val in zip(cx.row, cx.col, cx.data)
    ]

    print(f"Total triples: {len(triples):,}")

    output = {
        "metadata": {
            "n_docs":         n_docs,
            "n_terms":        n_terms,
            "n_triples":      len(triples),
            "ngram_range":    [1, 3],
            "max_df":         0.90,
            "min_df":         2,
            "sublinear_tf":   True,
            "preprocessing":  "biomed_regex_lemmatize",
        },
        "vocabulary": vocab.tolist(),
        "doc_index":  doc_index,
        "triples":    triples,
    }

    with open(TRIPLES_JSON, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False)

    size_mb = TRIPLES_JSON.stat().st_size / (1024 ** 2)
    print(f"\nSaved → data/ir/{TRIPLES_JSON.name}  ({size_mb:.2f} MB)")
    print("\nSample triples (first 10):")
    for t in triples[:10]:
        print(f"  doc_id={t['doc_id']:>6}  term={t['term']:<35}  score={t['score']:.4f}")


Converting sparse TF-IDF matrix to (doc_id, term, score) triples...
Total triples: 12,838,740

Saved → /home/pfanyka/Desktop/MASTERS/sem_2/IPA/ELIQSIR/data/ir/tfidf_triples.json  (787.44 MB)

Sample triples (first 10):
  doc_id=     1  term=exploring                            score=0.0585
  doc_id=     1  term=new                                  score=0.0239
  doc_id=     1  term=frontier                             score=0.0880
  doc_id=     1  term=cancer                               score=0.0268
  doc_id=     1  term=treatment                            score=0.0273
  doc_id=     1  term=targeting                            score=0.0409
  doc_id=     1  term=ubiquitin                            score=0.0912
  doc_id=     1  term=like                                 score=0.0440
  doc_id=     1  term=activating                           score=0.0961
  doc_id=     1  term=enzyme                               score=0.0521


---
## Summary & Output Verification


In [ ]:
# Reload and verify both output files
print("=== Output File Verification ===\n")

# --- ir_documents.json ---
with open(DOCUMENTS_JSON, 'r') as f:
    docs_loaded = json.load(f)

print("ir_documents.json")
print(f"  Documents      : {len(docs_loaded):,}")
print(f"  Keys per doc   : {list(docs_loaded[0].keys())}")
sample = docs_loaded[0]
print(f"  Sample doc_id  : {sample['doc_id']}")
print(f"  drug_chembl_ids: {sample['drug_chembl_ids'][:5]}")
print(f"  uniprot_ids    : {sample['uniprot_ids'][:5]}")
print(f"  gene_names     : {sample['gene_names'][:5]}")

print()

# --- tfidf_triples.json ---
with open(TRIPLES_JSON, 'r') as f:
    triples_loaded = json.load(f)

meta = triples_loaded['metadata']
print("tfidf_triples.json")
print(f"  Documents   : {meta['n_docs']:,}")
print(f"  Vocabulary  : {meta['n_terms']:,} terms")
print(f"  Triples     : {meta['n_triples']:,}")
print(f"  Matrix density: {100 * meta['n_triples'] / (meta['n_docs'] * meta['n_terms']):.3f}%")
print(f"  Config      : ngram_range={meta['ngram_range']}, "
      f"min_df={meta['min_df']}, max_df={meta['max_df']}")
print("\n  Sample triples:")
for t in triples_loaded['triples'][:5]:
    print(f"    {t}")


=== Output File Verification ===

ir_documents.json
  Documents      : 35,561
  Keys per doc   : ['doc_id', 'pubmed_id', 'article_title', 'abstract', 'journal', 'year', 'authors', 'drug_chembl_ids', 'drug_names', 'uniprot_ids', 'gene_names', 'protein_names', 'protein_families', 'document_text']
  Sample doc_id  : 1
  drug_chembl_ids: ['CHEMBL1173445', 'CHEMBL2017005', 'CHEMBL2322194', 'CHEMBL2322195', 'CHEMBL2322196']
  uniprot_ids    : ['A0AVT1', 'O95352', 'P41226', 'Q13564', 'Q9UBE0']
  gene_names     : ['ATG7', 'APG7L', 'NAE1', 'APPBP1', 'HPP1']

